In [ ]:
-- 실행 컨텍스트 설정
USE ROLE ACCOUNTADMIN;
USE DATABASE DEMO;
USE SCHEMA ML_DEMO;

# Module 7: ML Observability
## 모델 모니터링 (Model Monitoring & Drift Detection)

---

## 학습 목표

이 모듈에서는 프로덕션 환경에서 배포된 ML 모델을 **관찰(Observe)**하는 방법을 학습합니다.

| 주제 | 내용 |
|------|------|
| **Model Monitoring** | 모델 성능 메트릭 추적 (정확도, F1) |
| **Data Drift 감지** | PSI 기반 피처 드리프트 탐지 |
| **ML Lineage** | 모델 → 데이터 소스 계보 추적 |

---

## 왜 ML Observability가 중요한가?

```
학습 시점                           현재 (프로덕션)
────────────────────────────────────────────────────
  데이터 분포 A  →  모델 학습  →  데이터 분포 B (드리프트!)
  정확도 92%                       정확도 73% (모델 부패!)
```

### 주요 문제:
1. **Data Drift**: 입력 피처의 분포가 시간이 지남에 따라 변함
2. **Concept Drift**: 피처와 레이블 간의 관계가 변함
3. **Model Decay**: 모델 성능이 점진적으로 저하됨
4. **Silent Failures**: 명확한 오류 없이 예측이 틀리기 시작함

Snowflake의 **ML Observability** 기능을 사용하면 이러한 문제를 조기에 감지하고 대응할 수 있습니다.

## 1. 환경 설정 (Setup)

Snowflake ML Observability에 필요한 라이브러리를 임포트하고 세션을 초기화합니다.

In [ ]:
# ── 핵심 라이브러리 임포트 ──────────────────────────────────────────────────
from snowflake.snowpark.context import get_active_session
from snowflake.ml.registry import Registry
import snowflake.snowpark.functions as F
import snowflake.snowpark.types as T

# ── 데이터 분석 / 시각화 ────────────────────────────────────────────────────
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import matplotlib.gridspec as gridspec
import warnings
warnings.filterwarnings('ignore')

# ── Snowflake 세션 초기화 ───────────────────────────────────────────────────
session = get_active_session()
session.use_database("DEMO")
session.use_schema("ML_DEMO")
session.use_warehouse("COMPUTE_WH")

print("✅ 세션 초기화 완료")
print(f"   Database : {session.get_current_database()}")
print(f"   Schema   : {session.get_current_schema()}")
print(f"   Warehouse: {session.get_current_warehouse()}")
print(f"   Role     : {session.get_current_role()}")
# ── 한글 폰트 설정 (Snowflake Workspaces Container Runtime) ─────────────
import subprocess, matplotlib, matplotlib.font_manager as fm
subprocess.run(['apt-get', 'install', '-y', 'fonts-nanum'],
               capture_output=True)  # NanumGothic 설치
fm._load_fontmanager(try_read_cache=False)
nanum = [f for f in fm.findSystemFonts() if 'Nanum' in f]
if nanum:
    fm.fontManager.addfont(nanum[0])
    matplotlib.rc('font', family='NanumGothic')
matplotlib.rc('axes', unicode_minus=False)  # 마이너스 기호 깨짐 방지


In [ ]:
# ── 등록된 모델 확인 ────────────────────────────────────────────────────────
reg = Registry(session=session, database_name="DEMO", schema_name="ML_DEMO")

model_list = reg.show_models()
print("📦 등록된 모델 목록:")
print(model_list)

# 모델 버전 확인
mv = reg.get_model("CUSTOMER_LTV_PREDICTOR").version("V1")
print("\n✅ 모델 로드 완료:", mv)

## 2. 추론 로그 테이블 준비 (Inference Log Table)

Model Monitor는 모델이 실제로 예측한 결과를 저장하는 **추론 로그 테이블**을 필요로 합니다.

### 테이블 스키마

| 컬럼명 | 타입 | 설명 |
|--------|------|------|
| `PREDICTED_LTV_ID` | VARCHAR | 예측 고유 ID |
| `CUSTOMER_KEY` | NUMBER | 고객 ID (TPC-H C_CUSTKEY) |
| `PREDICTED_LTV` | NUMBER | 모델 예측값 (0/1) |
| `ACTUAL_LABEL` | NUMBER | 실제 레이블 (ground truth) |
| `PREDICTED_LTV_TIME` | TIMESTAMP | 예측 수행 시각 |
| `ACCTBAL` | FLOAT | 계좌 잔액 |
| `TOTAL_ORDERS` | NUMBER | 총 주문 수 |
| `AVG_ORDER_AMOUNT` | FLOAT | 평균 주문 금액 |
| `MAX_ORDER_AMOUNT` | FLOAT | 최대 주문 금액 |
| `AVG_DISCOUNT` | FLOAT | 평균 할인율 |
| `TOTAL_REVENUE` | FLOAT | 총 매출 |
| `NATION_CODE` | NUMBER | 국가 코드 |

> **드리프트 시뮬레이션**: 최근 10일간의 데이터에 의도적으로 다른 분포를 적용하여 피처 드리프트를 시뮬레이션합니다.

In [ ]:
# ── 추론 로그 테이블 생성 ───────────────────────────────────────────────────
session.sql("""
CREATE OR REPLACE TABLE DEMO.ML_DEMO.INFERENCE_LOG (
    PREDICTED_LTV_ID     VARCHAR(50)        NOT NULL,
    CUSTOMER_KEY      NUMBER(38,0),
    PREDICTED_LTV        NUMBER(1,0),
    ACTUAL_LABEL      NUMBER(1,0),
    PREDICTED_LTV_TIME   TIMESTAMP_NTZ,
    ACCTBAL           FLOAT,
    TOTAL_ORDERS      NUMBER(38,0),
    AVG_ORDER_AMOUNT  FLOAT,
    MAX_ORDER_AMOUNT  FLOAT,
    AVG_DISCOUNT      FLOAT,
    TOTAL_REVENUE     FLOAT,
    NATION_CODE       NUMBER(38,0)
)
""").collect()

print("✅ INFERENCE_LOG 테이블 생성 완료")

In [ ]:
# ── 시뮬레이션 데이터 생성 (최근 30일) ─────────────────────────────────────
np.random.seed(42)

n_records = 3000
# 날짜 범위: 최근 30일
base_ts = pd.Timestamp('now') - pd.Timedelta(days=30)
timestamps = [base_ts + pd.Timedelta(seconds=int(i * 86400 * 30 / n_records))
              for i in range(n_records)]

# ── 정상 구간 (처음 20일, 0~2000건) ────────────────────────────────────────
n_normal = 2000
normal_data = {
    'PREDICTED_LTV_ID'   : [f'PRED_{i:06d}' for i in range(n_normal)],
    'CUSTOMER_KEY'    : np.random.randint(1, 15001, n_normal),
    'ACCTBAL'         : np.random.normal(4500, 2800, n_normal).clip(0),
    'TOTAL_ORDERS'    : np.random.poisson(7, n_normal).clip(1),
    'AVG_ORDER_AMOUNT': np.random.normal(180000, 80000, n_normal).clip(10000),
    'MAX_ORDER_AMOUNT': np.random.normal(450000, 150000, n_normal).clip(50000),
    'AVG_DISCOUNT'    : np.random.beta(2, 8, n_normal),
    'TOTAL_REVENUE'   : np.random.normal(1200000, 600000, n_normal).clip(50000),
    'NATION_CODE'     : np.random.randint(0, 25, n_normal),
    'PREDICTED_LTV_TIME' : timestamps[:n_normal],
}

# ── 드리프트 구간 (최근 10일, 2000~3000건) ──────────────────────────────────
# 드리프트: ACCTBAL 급격히 감소, TOTAL_ORDERS 증가, AVG_DISCOUNT 급증
n_drift = 1000
drift_data = {
    'PREDICTED_LTV_ID'   : [f'PRED_{i:06d}' for i in range(n_normal, n_records)],
    'CUSTOMER_KEY'    : np.random.randint(1, 15001, n_drift),
    'ACCTBAL'         : np.random.normal(1200, 800, n_drift).clip(0),     # 급감 (drift!)
    'TOTAL_ORDERS'    : np.random.poisson(15, n_drift).clip(1),           # 급증 (drift!)
    'AVG_ORDER_AMOUNT': np.random.normal(90000, 40000, n_drift).clip(5000), # 감소 (drift!)
    'MAX_ORDER_AMOUNT': np.random.normal(300000, 100000, n_drift).clip(30000),
    'AVG_DISCOUNT'    : np.random.beta(6, 4, n_drift),                   # 할인율 급증 (drift!)
    'TOTAL_REVENUE'   : np.random.normal(800000, 400000, n_drift).clip(30000),
    'NATION_CODE'     : np.random.randint(0, 25, n_drift),
    'PREDICTED_LTV_TIME' : timestamps[n_normal:],
}

# 정상 + 드리프트 데이터 병합
all_keys = ['PREDICTED_LTV_ID','CUSTOMER_KEY','ACCTBAL','TOTAL_ORDERS',
            'AVG_ORDER_AMOUNT','MAX_ORDER_AMOUNT','AVG_DISCOUNT',
            'TOTAL_REVENUE','NATION_CODE','PREDICTED_LTV_TIME']

rows = []
for src in [normal_data, drift_data]:
    n = len(src['PREDICTED_LTV_ID'])
    for i in range(n):
        acctbal  = float(src['ACCTBAL'][i])
        orders   = int(src['TOTAL_ORDERS'][i])
        avg_amt  = float(src['AVG_ORDER_AMOUNT'][i])
        revenue  = float(src['TOTAL_REVENUE'][i])
        discount = float(src['AVG_DISCOUNT'][i])
        # 간단한 규칙 기반 레이블 (실제 ground truth 모사)
        prob_high = 1 / (1 + np.exp(-(0.0003*acctbal + 0.05*orders - 0.000001*avg_amt - 2.5)))
        actual   = int(np.random.random() < prob_high)
        # 드리프트 구간에서는 모델 예측 오류율 증가
        is_drift = src is drift_data
        noise    = 0.25 if is_drift else 0.08
        pred     = actual if np.random.random() > noise else 1 - actual
        rows.append((
            src['PREDICTED_LTV_ID'][i],
            int(src['CUSTOMER_KEY'][i]),
            pred, actual,
            src['PREDICTED_LTV_TIME'][i].to_pydatetime(),
            acctbal, orders, avg_amt,
            float(src['MAX_ORDER_AMOUNT'][i]),
            discount, revenue,
            int(src['NATION_CODE'][i])
        ))

print(f"✅ 시뮬레이션 데이터 생성 완료: {len(rows):,}건")
print(f"   정상 구간: {n_normal:,}건 (20일)")
print(f"   드리프트 구간: {n_drift:,}건 (최근 10일)")

In [ ]:
# ── Snowpark DataFrame으로 변환 후 테이블에 삽입 ────────────────────────────
schema = T.StructType([
    T.StructField("PREDICTED_LTV_ID",     T.StringType()),
    T.StructField("CUSTOMER_KEY",      T.LongType()),
    T.StructField("PREDICTED_LTV",        T.LongType()),
    T.StructField("ACTUAL_LABEL",      T.LongType()),
    T.StructField("PREDICTED_LTV_TIME",   T.TimestampType()),
    T.StructField("ACCTBAL",           T.DoubleType()),
    T.StructField("TOTAL_ORDERS",      T.LongType()),
    T.StructField("AVG_ORDER_AMOUNT",  T.DoubleType()),
    T.StructField("MAX_ORDER_AMOUNT",  T.DoubleType()),
    T.StructField("AVG_DISCOUNT",      T.DoubleType()),
    T.StructField("TOTAL_REVENUE",     T.DoubleType()),
    T.StructField("NATION_CODE",       T.LongType()),
])

# 배치로 삽입 (메모리 효율)
BATCH_SIZE = 500
total_inserted = 0
for start in range(0, len(rows), BATCH_SIZE):
    batch = rows[start:start + BATCH_SIZE]
    sdf = session.create_dataframe(batch, schema=schema)
    sdf.write.mode("append").save_as_table("DEMO.ML_DEMO.INFERENCE_LOG")
    total_inserted += len(batch)

print(f"✅ INFERENCE_LOG 삽입 완료: {total_inserted:,}건")

# 테이블 확인
count = session.table("DEMO.ML_DEMO.INFERENCE_LOG").count()
print(f"   테이블 총 레코드 수: {count:,}")

In [ ]:
# ── 데이터 미리보기 ─────────────────────────────────────────────────────────
preview_df = session.table("DEMO.ML_DEMO.INFERENCE_LOG") \
    .order_by(F.col("PREDICTED_LTV_TIME")) \
    .limit(5) \
    .to_pandas()

print("📋 INFERENCE_LOG 상위 5건:")
print(preview_df.to_string(index=False))

# 레이블 분포 확인
label_dist = session.table("DEMO.ML_DEMO.INFERENCE_LOG") \
    .group_by("ACTUAL_LABEL") \
    .count() \
    .to_pandas()
print("\n📊 실제 레이블 분포:")
print(label_dist)

## 3. Model Monitor 생성

Snowflake **Model Monitor**는 추론 로그 테이블을 기반으로 모델 성능을 지속적으로 추적합니다.

`CREATE MODEL MONITOR` SQL DDL로 생성하며, Python 패키지 의존성이 없습니다.

### CREATE MODEL MONITOR 주요 파라미터

| 파라미터 | 값 | 설명 |
|----------|-----|------|
| `MODEL` | `CUSTOMER_LTV_PREDICTOR` | 모니터링할 모델명 |
| `VERSION` | `'V1'` | 모델 버전 |
| `FUNCTION` | `'PREDICT'` | 모델 함수명 |
| `SOURCE` | `INFERENCE_LOG` | 추론 로그 테이블 |
| `TIMESTAMP_COLUMN` | `PREDICTED_LTV_TIME` | 예측 시각 컬럼 (TIMESTAMP_NTZ) |
| `PREDICTION_SCORE_COLUMNS` | `('PREDICTED_LTV')` | 모델 예측값 컬럼 |
| `ACTUAL_SCORE_COLUMNS` | `('ACTUAL_LABEL')` | 실제 레이블 컬럼 |
| `REFRESH_INTERVAL` | `'1 day'` | 모니터 갱신 주기 |
| `AGGREGATION_WINDOW` | `'1 day'` | 집계 단위 |

> **참고**: 모니터는 모델과 동일한 스키마에 생성해야 합니다.


In [ ]:
# ── Model Monitor 생성 전 권한 확인 ──────────────────────────────────────────
# CREATE MODEL MONITOR 권한 필요: CREATE MODEL MONITOR ON SCHEMA
session.sql("SHOW MODEL MONITORS IN SCHEMA DEMO.ML_DEMO").show()


In [ ]:
# ── Model Monitor 생성 (SQL DDL) ─────────────────────────────────────────────
# 기존 모니터 삭제 (있으면)
try:
    session.sql("DROP MODEL MONITOR IF EXISTS DEMO.ML_DEMO.LTV_MODEL_MONITOR").collect()
    print("기존 모니터 삭제 완료")
except Exception:
    pass

# Model Monitor 생성
session.sql("""
CREATE MODEL MONITOR DEMO.ML_DEMO.LTV_MODEL_MONITOR WITH
    MODEL = DEMO.ML_DEMO.CUSTOMER_LTV_PREDICTOR
    VERSION = 'V1'
    FUNCTION = 'PREDICT'
    SOURCE = DEMO.ML_DEMO.INFERENCE_LOG
    WAREHOUSE = COMPUTE_WH
    REFRESH_INTERVAL = '1 day'
    AGGREGATION_WINDOW = '1 day'
    TIMESTAMP_COLUMN = PREDICTED_LTV_TIME
    ID_COLUMNS = ('PREDICTED_LTV_ID')
    PREDICTION_SCORE_COLUMNS = ('PREDICTED_LTV')
    ACTUAL_SCORE_COLUMNS = ('ACTUAL_LABEL')
""").collect()

print("Model Monitor 생성 완료!")
print(f"   모니터 이름: LTV_MODEL_MONITOR")
print(f"   대상 테이블: DEMO.ML_DEMO.INFERENCE_LOG")
print(f"   모델: CUSTOMER_LTV_PREDICTOR / V1")
print()

# 생성 확인
session.sql("SHOW MODEL MONITORS IN SCHEMA DEMO.ML_DEMO").show()


## 4. 성능 메트릭 모니터링 (Performance Metrics)

시간 경과에 따른 모델 성능 변화를 추적합니다.

### 주요 메트릭
- **RMSE**: 예측 오차 크기 (낮을수록 좋음)
- **Precision**: 양성 예측 중 실제 양성 비율 `TP / (TP+FP)`
- **Recall**: 실제 양성 중 올바르게 예측한 비율 `TP / (TP+FN)`
- **F1 Score**: Precision과 Recall의 조화평균 `2 * P * R / (P+R)`

In [ ]:
# ── 성능 메트릭 조회 (MODEL_MONITOR_PERFORMANCE_METRIC) ──────────────────────
try:
    metrics_df = session.sql("""
    SELECT *
    FROM TABLE(MODEL_MONITOR_PERFORMANCE_METRIC(
        'LTV_MODEL_MONITOR',
        'RMSE',
        'DAY',
        DATEADD('day', -30, CURRENT_TIMESTAMP())::TIMESTAMP_NTZ,
        CURRENT_TIMESTAMP()::TIMESTAMP_NTZ
    ))
    ORDER BY 1
    """).to_pandas()
    print("MODEL_MONITOR_PERFORMANCE_METRIC 조회 성공")
except Exception as e:
    print(f"Monitor 메트릭 함수 호출 실패 ({type(e).__name__}): 수동 계산으로 fallback")
    # Fallback: INFERENCE_LOG에서 직접 계산
    metrics_sql = """
    SELECT
        DATE_TRUNC('DAY', PREDICTED_LTV_TIME)            AS METRIC_DATE,
        COUNT(*)                                         AS TOTAL_PREDS,
        ROUND(
            SUM(CASE WHEN PREDICTED_LTV=1 AND ACTUAL_LABEL=1 THEN 1 ELSE 0 END)
            / NULLIF(SUM(CASE WHEN PREDICTED_LTV=1 THEN 1 ELSE 0 END),0), 4
        ) AS PRECISION_VAL,
        ROUND(
            SUM(CASE WHEN PREDICTED_LTV=1 AND ACTUAL_LABEL=1 THEN 1 ELSE 0 END)
            / NULLIF(SUM(CASE WHEN ACTUAL_LABEL=1 THEN 1 ELSE 0 END),0), 4
        ) AS RECALL_VAL
    FROM DEMO.ML_DEMO.INFERENCE_LOG
    GROUP BY 1
    ORDER BY 1
    """
    metrics_df = session.sql(metrics_sql).to_pandas()
    # F1 계산
    p = metrics_df['PRECISION_VAL'].fillna(0)
    r = metrics_df['RECALL_VAL'].fillna(0)
    metrics_df['F1_SCORE'] = 2 * p * r / (p + r + 1e-9)

print("\n일별 성능 메트릭 (상위 10일):")
print(metrics_df.head(10).to_string(index=False))


In [ ]:
# ── 성능 트렌드 시각화 ──────────────────────────────────────────────────────
fig, axes = plt.subplots(2, 1, figsize=(14, 9), facecolor="#0d1117")
fig.patch.set_facecolor("#0d1117")

date_col = [c for c in metrics_df.columns if "DATE" in c.upper() or "TIME" in c.upper()][0]
x = pd.to_datetime(metrics_df[date_col])

# 드리프트 시작 지점 (최근 10일)
drift_start = x.max() - pd.Timedelta(days=10)

# ── 플롯 1: Precision ────────────────────────────────────────────────────
ax1 = axes[0]
ax1.set_facecolor("#161b22")
ax1.plot(x, metrics_df["PRECISION_VAL"], color="#58a6ff", linewidth=2.5,
         marker="o", markersize=4, label="Precision")
ax1.axhline(y=0.85, color="#f85149", linestyle="--", linewidth=1.5,
            label="임계값 (0.85)")
ax1.axvline(x=drift_start, color="#ff7b72", linestyle=":", linewidth=2,
            label="드리프트 시작")
ax1.set_title("일별 Precision 추이", color="white", fontsize=13, pad=10)
ax1.set_ylabel("Precision", color="#8b949e")
ax1.set_ylim(0.3, 1.05)
ax1.tick_params(colors="#8b949e")
ax1.spines["bottom"].set_color("#30363d")
ax1.spines["top"].set_color("#30363d")
ax1.spines["left"].set_color("#30363d")
ax1.spines["right"].set_color("#30363d")
ax1.legend(facecolor="#21262d", labelcolor="white", fontsize=9)
ax1.grid(axis="y", color="#21262d", alpha=0.7)

# ── 플롯 2: F1 Score + Recall ────────────────────────────────────────────
ax2 = axes[1]
ax2.set_facecolor("#161b22")
f1_col = "F1_SCORE" if "F1_SCORE" in metrics_df.columns else None
if f1_col:
    ax2.plot(x, metrics_df[f1_col], color="#3fb950", linewidth=2.5,
             marker="s", markersize=4, label="F1 Score")
ax2.plot(x, metrics_df["RECALL_VAL"], color="#ffa657", linewidth=1.5,
         linestyle="--", label="Recall")
ax2.axvline(x=drift_start, color="#ff7b72", linestyle=":", linewidth=2,
            label="드리프트 시작")
ax2.set_title("F1 Score / Recall 추이", color="white", fontsize=13, pad=10)
ax2.set_ylabel("Score", color="#8b949e")
ax2.set_xlabel("날짜", color="#8b949e")
ax2.set_ylim(0.3, 1.05)
ax2.tick_params(colors="#8b949e")
ax2.spines["bottom"].set_color("#30363d")
ax2.spines["top"].set_color("#30363d")
ax2.spines["left"].set_color("#30363d")
ax2.spines["right"].set_color("#30363d")
ax2.legend(facecolor="#21262d", labelcolor="white", fontsize=9)
ax2.grid(axis="y", color="#21262d", alpha=0.7)

plt.tight_layout(pad=2.0)
plt.suptitle("Model Performance Over Time — CUSTOMER_LTV_PREDICTOR V1",
             color="#e6edf3", fontsize=14, fontweight="bold", y=1.01)
plt.show()
print("\n드리프트 구간(최근 10일)에서 성능 변화 관찰!")

## 5. 데이터 드리프트 감지 (Feature Drift Detection)

### PSI (Population Stability Index)란?

PSI는 두 분포 간의 차이를 측정하는 지표입니다:

$$PSI = \sum_{i=1}^{n} (A_i - E_i) \times \ln\left(\frac{A_i}{E_i}\right)$$

여기서:
- $A_i$ = 실제(현재) 분포의 비율
- $E_i$ = 기대(기준) 분포의 비율

### PSI 해석 기준

| PSI 값 | 판정 | 조치 |
|--------|------|------|
| `< 0.10` | 안정 (Stable) | 모니터링 유지 |
| `0.10 ~ 0.20` | 경미한 변화 (Slight Shift) | 주의 필요 |
| `> 0.20` | 심각한 드리프트 (Significant Drift) | 재학습 권장 |

In [ ]:
# ── 피처 드리프트 감지 (MODEL_MONITOR_DRIFT_METRIC) ──────────────────────────
FEATURE_COLS = ['ACCTBAL','TOTAL_ORDERS','AVG_ORDER_AMOUNT',
                'MAX_ORDER_AMOUNT','AVG_DISCOUNT','TOTAL_REVENUE']

try:
    # Monitor 드리프트 메트릭 함수로 조회 (피처별 PSI)
    drift_results = []
    for feat in FEATURE_COLS:
        df = session.sql(f"""
        SELECT *
        FROM TABLE(MODEL_MONITOR_DRIFT_METRIC(
            'LTV_MODEL_MONITOR',
            'PSI',
            '{feat}',
            'DAY',
            DATEADD('day', -30, CURRENT_TIMESTAMP())::TIMESTAMP_NTZ,
            CURRENT_TIMESTAMP()::TIMESTAMP_NTZ
        ))
        """).to_pandas()
        if not df.empty:
            drift_results.append(df)
    if drift_results:
        drift_df = pd.concat(drift_results, ignore_index=True)
        print("MODEL_MONITOR_DRIFT_METRIC 조회 성공")
    else:
        raise ValueError("No drift data returned")
except Exception as e:
    print(f"Monitor 드리프트 함수 호출 실패 ({type(e).__name__}): 수동 PSI 계산으로 fallback")

    # Fallback: 수동 PSI 계산
    def compute_psi(reference, current, n_bins=10):
        """Population Stability Index 계산"""
        eps = 1e-6
        bins = np.percentile(reference, np.linspace(0, 100, n_bins + 1))
        bins[0] = -np.inf
        bins[-1] = np.inf
        ref_pct = np.histogram(reference, bins=bins)[0] / len(reference)
        cur_pct = np.histogram(current,   bins=bins)[0] / len(current)
        ref_pct = np.where(ref_pct == 0, eps, ref_pct)
        cur_pct = np.where(cur_pct == 0, eps, cur_pct)
        return float(np.sum((cur_pct - ref_pct) * np.log(cur_pct / ref_pct)))

    cutoff = (pd.Timestamp('now') - pd.Timedelta(days=10)).strftime('%Y-%m-%d')

    ref_sdf = session.table("DEMO.ML_DEMO.INFERENCE_LOG") \
        .filter(F.col("PREDICTED_LTV_TIME") < F.lit(cutoff)) \
        .select(*FEATURE_COLS)
    cur_sdf = session.table("DEMO.ML_DEMO.INFERENCE_LOG") \
        .filter(F.col("PREDICTED_LTV_TIME") >= F.lit(cutoff)) \
        .select(*FEATURE_COLS)

    ref_pd = ref_sdf.to_pandas()
    cur_pd = cur_sdf.to_pandas()

    psi_results = []
    for feat in FEATURE_COLS:
        psi_val = compute_psi(ref_pd[feat].dropna().values,
                              cur_pd[feat].dropna().values)
        status = 'HIGH DRIFT' if psi_val > 0.2 else \
                 'SLIGHT SHIFT' if psi_val > 0.1 else 'STABLE'
        psi_results.append({
            'FEATURE': feat,
            'PSI'    : round(psi_val, 4),
            'STATUS' : status
        })
    drift_df = pd.DataFrame(psi_results).sort_values('PSI', ascending=False)

print("\n피처별 PSI (Population Stability Index):")
print(drift_df.to_string(index=False))


In [ ]:
# ── 피처 드리프트 시각화 ────────────────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(16, 6), facecolor='#0d1117')
fig.patch.set_facecolor('#0d1117')

# 색상 매핑
color_map = {
    'HIGH DRIFT'  : '#f85149',
    'SLIGHT SHIFT': '#ffa657',
    'STABLE'      : '#3fb950'
}
bar_colors = [color_map.get(s, '#58a6ff')
              for s in drift_df.get('STATUS', ['STABLE']*len(drift_df))]

# ── PSI 막대 차트 ────────────────────────────────────────────────────────────
ax1 = axes[0]
ax1.set_facecolor('#161b22')
features = drift_df['FEATURE'].tolist()
psi_vals = drift_df['PSI'].tolist()

bars = ax1.barh(features, psi_vals, color=bar_colors, edgecolor='#30363d',
                height=0.6)
ax1.axvline(x=0.10, color='#ffa657', linestyle='--', linewidth=1.5,
            label='경고 임계값 (0.10)')
ax1.axvline(x=0.20, color='#f85149', linestyle='--', linewidth=1.5,
            label='위험 임계값 (0.20)')

for bar, val in zip(bars, psi_vals):
    ax1.text(val + 0.005, bar.get_y() + bar.get_height()/2,
             f'{val:.4f}', va='center', color='white', fontsize=9)

ax1.set_title('피처별 PSI (Population Stability Index)',
              color='white', fontsize=12, pad=10)
ax1.set_xlabel('PSI 값', color='#8b949e')
ax1.tick_params(colors='#8b949e')
for sp in ax1.spines.values():
    sp.set_color('#30363d')
ax1.legend(facecolor='#21262d', labelcolor='white', fontsize=9)
ax1.grid(axis='x', color='#21262d', alpha=0.7)

# ── 분포 비교: 가장 드리프트가 큰 피처 ────────────────────────────────────
ax2 = axes[1]
ax2.set_facecolor('#161b22')
top_feat = features[0]  # PSI 가장 높은 피처

ref_vals = ref_pd[top_feat].dropna().values
cur_vals = cur_pd[top_feat].dropna().values

ax2.hist(ref_vals, bins=30, alpha=0.6, color='#58a6ff',
         label='기준 분포 (20일 전)', density=True, edgecolor='none')
ax2.hist(cur_vals, bins=30, alpha=0.6, color='#f85149',
         label='현재 분포 (최근 10일)', density=True, edgecolor='none')
ax2.set_title(f'분포 비교: {top_feat}\n(PSI={psi_vals[0]:.4f} — HIGH DRIFT)',
              color='white', fontsize=12, pad=10)
ax2.set_xlabel(top_feat, color='#8b949e')
ax2.set_ylabel('Density', color='#8b949e')
ax2.tick_params(colors='#8b949e')
for sp in ax2.spines.values():
    sp.set_color('#30363d')
ax2.legend(facecolor='#21262d', labelcolor='white', fontsize=10)

# 범례 (드리프트 등급)
legend_patches = [
    mpatches.Patch(color='#f85149', label='HIGH DRIFT (PSI > 0.20)'),
    mpatches.Patch(color='#ffa657', label='SLIGHT SHIFT (0.10-0.20)'),
    mpatches.Patch(color='#3fb950', label='STABLE (PSI < 0.10)'),
]
ax1.legend(handles=legend_patches + ax1.get_legend_handles_labels()[0][:2],
           facecolor='#21262d', labelcolor='white', fontsize=8, loc='lower right')

plt.tight_layout(pad=2.0)
plt.suptitle('Feature Drift Detection — CUSTOMER_LTV_PREDICTOR V1',
             color='#e6edf3', fontsize=13, fontweight='bold', y=1.02)
plt.savefig('/tmp/08_feature_drift.png', dpi=150, bbox_inches='tight',
            facecolor='#0d1117')
plt.show()

high_drift_feats = drift_df[drift_df.get('STATUS','') == 'HIGH DRIFT']['FEATURE'].tolist()
if high_drift_feats:
    print(f"\n🚨 HIGH DRIFT 감지 피처: {high_drift_feats}")
    print("   → 모델 재학습을 강력히 권고합니다!")

## 6. 모델 드리프트 알림 설정 (Drift Alerts)

Snowflake의 **Alerts** 기능을 활용하여 모델 성능이 임계값 이하로 떨어지면 자동으로 알림을 보낼 수 있습니다.

### 알림 아키텍처

```
INFERENCE_LOG 테이블
      ↓  (주기적 집계)
MODEL_METRICS_DAILY 테이블
      ↓  (Snowflake Alert)
    조건: RMSE > 150,000 (USD)
      ↓  (트리거)
알림: Email / Slack / PagerDuty
```

### Snowsight 모니터링 대시보드
- **Snowsight > Monitoring > Model Monitor** 메뉴에서 시각적 대시보드 제공
- 실시간 정확도 차트, 피처 드리프트 히트맵, 예측 분포 확인 가능

In [ ]:
# ── 일별 집계 뷰 생성 ───────────────────────────────────────────────────────
session.sql("""
CREATE OR REPLACE VIEW DEMO.ML_DEMO.MODEL_METRICS_DAILY AS
SELECT
    DATE_TRUNC('DAY', PREDICTED_LTV_TIME)            AS METRIC_DATE,
    COUNT(*)                                       AS TOTAL_PREDICTED_LTVS,
    ROUND(AVG(CASE WHEN PREDICTED_LTV = ACTUAL_LABEL
                   THEN 1.0 ELSE 0.0 END), 4)    AS DAILY_RMSE,
    ROUND(
        SUM(CASE WHEN PREDICTED_LTV=1 AND ACTUAL_LABEL=1 THEN 1 ELSE 0 END)
        / NULLIF(SUM(CASE WHEN PREDICTED_LTV=1 THEN 1 ELSE 0 END), 0), 4
    )                                              AS DAILY_PRECISION,
    ROUND(
        SUM(CASE WHEN PREDICTED_LTV=1 AND ACTUAL_LABEL=1 THEN 1 ELSE 0 END)
        / NULLIF(SUM(CASE WHEN ACTUAL_LABEL=1 THEN 1 ELSE 0 END), 0), 4
    )                                              AS DAILY_RECALL
FROM DEMO.ML_DEMO.INFERENCE_LOG
GROUP BY 1
""").collect()

print("✅ MODEL_METRICS_DAILY 뷰 생성 완료")

In [ ]:
# ── 성능 저하 알림 Alert 생성 ──────────────────────────────────────────────
# (실제 환경에서는 notification integration 연결 필요)
alert_sql = """
CREATE OR REPLACE ALERT DEMO.ML_DEMO.MODEL_RMSE_ALERT
    WAREHOUSE = COMPUTE_WH
    SCHEDULE  = '1 HOUR'
    IF (EXISTS (
        SELECT 1
        FROM   DEMO.ML_DEMO.MODEL_METRICS_DAILY
        WHERE  METRIC_DATE = CURRENT_DATE - 1
          AND  DAILY_RMSE < 0.85
    ))
    THEN CALL SYSTEM$SEND_EMAIL(
        'MY_EMAIL_INTEGRATION',
        'ml-ops@example.com',
        '[ALERT] CUSTOMER_LTV_PREDICTOR 성능 저하 감지',
        'daily_rmse가 0.85 임계값 미달입니다. 모델 재학습을 검토하세요.'
    )
"""
print("📋 Alert 생성 SQL (실제 환경에서 실행):")
print(alert_sql)

# PSI 기반 드리프트 알림
drift_alert_concept = """
-- PSI 드리프트 알림 (개념)
CREATE OR REPLACE ALERT DEMO.ML_DEMO.FEATURE_DRIFT_ALERT
    WAREHOUSE = COMPUTE_WH
    SCHEDULE  = 'USING CRON 0 9 * * * UTC'   -- 매일 오전 9시
    IF (EXISTS (
        SELECT 1 FROM DEMO.ML_DEMO.FEATURE_PSI_DAILY
        WHERE METRIC_DATE = CURRENT_DATE - 1
          AND MAX_PSI > 0.20
    ))
    THEN CALL SYSTEM$SEND_EMAIL(...);
"""
print("\n📋 피처 드리프트 Alert 개념:")
print(drift_alert_concept)

## 8. 모니터링 대시보드 요약 (Monitoring Dashboard Summary)

성능 메트릭과 피처 드리프트를 하나의 대시보드 뷰로 통합합니다.

### 재학습 트리거 기준 (Retraining Criteria)

| 조건 | 임계값 | 조치 |
|------|--------|------|
| 정확도 하락 | `< 0.85` | 즉시 재학습 트리거 |
| F1 Score 하락 | `< 0.80` | 즉시 재학습 트리거 |
| HIGH DRIFT 피처 수 | `>= 2개` | 재학습 권고 |
| 최대 PSI | `> 0.30` | 즉시 재학습 트리거 |

In [ ]:
# ── 통합 모니터링 대시보드 ──────────────────────────────────────────────────
fig = plt.figure(figsize=(18, 12), facecolor='#0d1117')
gs  = gridspec.GridSpec(2, 2, figure=fig, hspace=0.45, wspace=0.35)

ax_acc   = fig.add_subplot(gs[0, :])   # 정확도 트렌드 (상단 2칸)
ax_psi   = fig.add_subplot(gs[0, 1])    # PSI 요약 (상단 우측)
ax_score = fig.add_subplot(gs[1, 1])    # 최종 점수카드 (하단 우측)

for ax in [ax_acc, ax_psi, ax_score]:
    ax.set_facecolor('#161b22')
    for sp in ax.spines.values(): sp.set_color('#30363d')

# ① 정확도 트렌드 ─────────────────────────────────────────────────────────
ax_acc.plot(x, metrics_df['PRECISION_VAL'], color='#58a6ff', linewidth=2.5,
            marker='o', markersize=3)
ax_acc.axhline(y=0.85, color='#f85149', linestyle='--', linewidth=1.5, alpha=0.8)
ax_acc.axvline(x=drift_start, color='#ff7b72', linestyle=':', linewidth=2)
ax_acc.fill_between(x, metrics_df['PRECISION_VAL'], 0.85,
                    where=metrics_df['PRECISION_VAL'] < 0.85,
                    alpha=0.25, color='#f85149')
ax_acc.set_title('Precision Trend', color='white', fontsize=11, pad=8)
ax_acc.set_ylabel('Precision', color='#8b949e', fontsize=9)
ax_acc.set_ylim(0.5, 1.05)
ax_acc.tick_params(colors='#8b949e', labelsize=8)
ax_acc.grid(axis='y', color='#21262d', alpha=0.7)
ax_acc.text(drift_start, 0.55, '← 드리프트\n  시작',
            color='#ff7b72', fontsize=8, ha='left')

# ② PSI 요약 ──────────────────────────────────────────────────────────────
psi_colors = [color_map.get(s, '#58a6ff') for s in
              drift_df.get('STATUS', ['STABLE']*len(drift_df))]
ax_psi.barh(drift_df['FEATURE'], drift_df['PSI'],
            color=psi_colors, edgecolor='#30363d', height=0.55)
ax_psi.axvline(x=0.20, color='#f85149', linestyle='--', linewidth=1)
ax_psi.axvline(x=0.10, color='#ffa657', linestyle='--', linewidth=1)
ax_psi.set_title('Feature Drift (PSI)', color='white', fontsize=11, pad=8)
ax_psi.set_xlabel('PSI', color='#8b949e', fontsize=9)
ax_psi.tick_params(colors='#8b949e', labelsize=8)

# ④ 점수카드 ──────────────────────────────────────────────────────────────
ax_score.axis('off')
latest_acc = metrics_df['PRECISION_VAL'].iloc[-1]
latest_f1  = metrics_df.get('F1_SCORE', metrics_df.get('F1',
             pd.Series([0]*len(metrics_df)))).iloc[-1]
n_high_drift = len(drift_df[drift_df.get('STATUS','') == 'HIGH DRIFT'])
max_psi      = drift_df['PSI'].max()

# 재학습 필요 여부
retrain_needed = (
    latest_acc  < 0.85 or
    latest_f1   < 0.80 or
    n_high_drift >= 2  or
    max_psi      > 0.30
)
status_color = '#f85149' if retrain_needed else '#3fb950'
status_text  = '재학습 필요!' if retrain_needed else '정상 운영 중'

score_items = [
    ('모델 상태', status_text, status_color, 14),
    ('─'*22, '', '#30363d', 9),
    ('현재 Precision', f'{latest_acc:.4f}',
     '#f85149' if latest_acc < 0.85 else '#3fb950', 11),
    ('현재 F1 Score', f'{latest_f1:.4f}',
     '#f85149' if latest_f1 < 0.80 else '#3fb950', 11),
    ('HIGH DRIFT 피처', f'{n_high_drift}개',
     '#f85149' if n_high_drift >= 2 else '#3fb950', 11),
    ('최대 PSI', f'{max_psi:.4f}',
     '#f85149' if max_psi > 0.20 else '#3fb950', 11),
    ('─'*22, '', '#30363d', 9),
    ('권고 조치', '재학습 실행' if retrain_needed else '모니터링 유지',
     status_color, 10),
]

y_pos = 0.95
for label, value, color, size in score_items:
    if value:
        ax_score.text(0.05, y_pos, label, transform=ax_score.transAxes,
                      color='#8b949e', fontsize=size-1, va='top')
        ax_score.text(0.95, y_pos, value, transform=ax_score.transAxes,
                      color=color, fontsize=size, va='top', ha='right',
                      fontweight='bold')
    else:
        ax_score.text(0.05, y_pos, label, transform=ax_score.transAxes,
                      color=color, fontsize=size, va='top')
    y_pos -= 0.12

ax_score.set_title('모니터링 점수카드', color='white', fontsize=11, pad=8)
ax_score.set_facecolor('#161b22')

fig.suptitle('ML Observability Dashboard — CUSTOMER_LTV_PREDICTOR V1',
             color='#e6edf3', fontsize=14, fontweight='bold', y=1.01)
plt.savefig('/tmp/08_monitoring_dashboard.png', dpi=150,
            bbox_inches='tight', facecolor='#0d1117')
plt.show()

print(f"\n{'🚨 재학습 권고!' if retrain_needed else '✅ 모델 정상 운영 중'}")
print(f"  - 현재 Precision : {latest_acc:.4f}  (임계값: 0.85)")
print(f"  - HIGH DRIFT 피처: {n_high_drift}개")
print(f"  - 최대 PSI       : {max_psi:.4f}")

In [ ]:
# ── 재학습 트리거 로직 ──────────────────────────────────────────────────────
def evaluate_retrain_need(rmse: float, r2: float,
                          n_high_drift: int, max_psi: float) -> dict:
    """재학습 필요 여부를 평가하고 이유를 반환합니다."""
    reasons = []
    if rmse > 150000:
        reasons.append(f'RMSE {rmse:,.0f} > 150,000')
    if f1 < 0.80:
        reasons.append(f'F1 Score {f1:.4f} < 0.80')
    if n_high_drift >= 2:
        reasons.append(f'HIGH DRIFT 피처 {n_high_drift}개 >= 2')
    if max_psi > 0.30:
        reasons.append(f'최대 PSI {max_psi:.4f} > 0.30')

    return {
        'retrain_needed': len(reasons) > 0,
        'reasons'       : reasons,
        'action'        : '즉시 재학습 실행' if len(reasons) > 0 else '모니터링 유지',
        'priority'      : 'HIGH' if len(reasons) >= 2 else
                          'MEDIUM' if len(reasons) == 1 else 'NORMAL'
    }

result = evaluate_retrain_need(latest_acc, latest_f1, n_high_drift, max_psi)
print("📋 재학습 평가 결과:")
print(f"  재학습 필요 : {result['retrain_needed']}")
print(f"  우선순위   : {result['priority']}")
print(f"  권고 조치  : {result['action']}")
if result['reasons']:
    print("  트리거 이유:")
    for r in result['reasons']:
        print(f"    • {r}")

## 9. ML Lineage — 데이터 계보 추적

Snowflake의 **ML Lineage** 기능을 통해 모델 → 피처 → 원본 테이블까지의 전체 데이터 흐름을 추적합니다.

### 계보 추적 방향

```
[UPSTREAM 방향]
CUSTOMER_LTV_PREDICTOR/V1
        ↑
CUSTOMER_FEATURES_FINAL (피처 테이블)
        ↑
CUSTOMER_FEATURES (중간 변환)
        ↑
  ┌─────────────────────────┐
  │  TPCH_SF1.CUSTOMER      │
  │  TPCH_SF1.ORDERS        │
  │  TPCH_SF1.LINEITEM      │
  │  TPCH_SF1.NATION        │
  └─────────────────────────┘
            (원본 TPC-H 테이블)
```

### GET_LINEAGE 함수 파라미터

| 파라미터 | 값 | 설명 |
|----------|-----|------|
| 객체 타입 | `'MODEL'` | 모델 계보 추적 |
| 객체 이름 | `'DEMO.ML_DEMO.CUSTOMER_LTV_PREDICTOR/V1'` | 대상 모델 |
| 방향 | `'UPSTREAM'` | 상위(소스) 방향 추적 |
| 재귀 | `TRUE` | 전체 체인 추적 |

In [ ]:
# ── ML Lineage 조회 ────────────────────────────────────────────────────────
lineage_sql = """
SELECT *
FROM TABLE(
    DEMO.ML_DEMO.GET_LINEAGE(
        'MODEL',
        'DEMO.ML_DEMO.CUSTOMER_LTV_PREDICTOR/V1',
        'UPSTREAM',
        TRUE
    )
)
"""

try:
    lineage_df = session.sql(lineage_sql).to_pandas()
    print("✅ ML Lineage 조회 완료")
    print(f"   계보 노드 수: {len(lineage_df)}")
    print(lineage_df.to_string(index=False))
    LINEAGE_SOURCE = "snowflake"
except Exception as e:
    print(f"ℹ️  GET_LINEAGE 미지원 환경 ({type(e).__name__}): 시뮬레이션 계보 사용")
    LINEAGE_SOURCE = "simulation"
    # 시뮬레이션 계보 데이터
    lineage_df = pd.DataFrame([
        {'SOURCE_OBJECT_DOMAIN': 'MODEL',  'SOURCE_OBJECT_NAME': 'DEMO.ML_DEMO.CUSTOMER_LTV_PREDICTOR',
         'SOURCE_OBJECT_VERSION': 'V1',    'TARGET_OBJECT_DOMAIN': None, 'TARGET_OBJECT_NAME': None,
         'DISTANCE': 0},
        {'SOURCE_OBJECT_DOMAIN': 'TABLE',  'SOURCE_OBJECT_NAME': 'DEMO.ML_DEMO.CUSTOMER_FEATURES_FINAL',
         'SOURCE_OBJECT_VERSION': None,    'TARGET_OBJECT_DOMAIN': 'MODEL', 'TARGET_OBJECT_NAME': 'CUSTOMER_LTV_PREDICTOR',
         'DISTANCE': 1},
        {'SOURCE_OBJECT_DOMAIN': 'TABLE',  'SOURCE_OBJECT_NAME': 'DEMO.ML_DEMO.CUSTOMER_FEATURES',
         'SOURCE_OBJECT_VERSION': None,    'TARGET_OBJECT_DOMAIN': 'TABLE', 'TARGET_OBJECT_NAME': 'CUSTOMER_FEATURES_FINAL',
         'DISTANCE': 2},
        {'SOURCE_OBJECT_DOMAIN': 'TABLE',  'SOURCE_OBJECT_NAME': 'SNOWFLAKE_SAMPLE_DATA.TPCH_SF1.CUSTOMER',
         'SOURCE_OBJECT_VERSION': None,    'TARGET_OBJECT_DOMAIN': 'TABLE', 'TARGET_OBJECT_NAME': 'CUSTOMER_FEATURES',
         'DISTANCE': 3},
        {'SOURCE_OBJECT_DOMAIN': 'TABLE',  'SOURCE_OBJECT_NAME': 'SNOWFLAKE_SAMPLE_DATA.TPCH_SF1.ORDERS',
         'SOURCE_OBJECT_VERSION': None,    'TARGET_OBJECT_DOMAIN': 'TABLE', 'TARGET_OBJECT_NAME': 'CUSTOMER_FEATURES',
         'DISTANCE': 3},
        {'SOURCE_OBJECT_DOMAIN': 'TABLE',  'SOURCE_OBJECT_NAME': 'SNOWFLAKE_SAMPLE_DATA.TPCH_SF1.LINEITEM',
         'SOURCE_OBJECT_VERSION': None,    'TARGET_OBJECT_DOMAIN': 'TABLE', 'TARGET_OBJECT_NAME': 'CUSTOMER_FEATURES',
         'DISTANCE': 3},
        {'SOURCE_OBJECT_DOMAIN': 'TABLE',  'SOURCE_OBJECT_NAME': 'SNOWFLAKE_SAMPLE_DATA.TPCH_SF1.NATION',
         'SOURCE_OBJECT_VERSION': None,    'TARGET_OBJECT_DOMAIN': 'TABLE', 'TARGET_OBJECT_NAME': 'CUSTOMER_FEATURES',
         'DISTANCE': 3},
    ])
    print("\n📋 시뮬레이션 계보 데이터:")
    print(lineage_df.to_string(index=False))

In [ ]:
# ── Lineage 그래프 시각화 ──────────────────────────────────────────────────
fig, ax = plt.subplots(figsize=(14, 8), facecolor='#0d1117')
ax.set_facecolor('#0d1117')
ax.axis('off')

# 노드 위치 정의 (레이어별)
nodes = {
    # 레이어 0: 모델
    'CUSTOMER_LTV_PREDICTOR\n(MODEL V1)': (7, 4),
    # 레이어 1: 피처 테이블
    'CUSTOMER_FEATURES_FINAL\n(DEMO.ML_DEMO)': (7, 3),
    # 레이어 2: 중간 변환
    'CUSTOMER_FEATURES\n(DEMO.ML_DEMO)': (7, 2),
    # 레이어 3: 원본 TPC-H
    'CUSTOMER\n(TPCH_SF1)': (3.5, 0.8),
    'ORDERS\n(TPCH_SF1)'  : (5.5, 0.8),
    'LINEITEM\n(TPCH_SF1)': (7.5, 0.8),
    'NATION\n(TPCH_SF1)'  : (9.5, 0.8),
    # 추론 로그
    'INFERENCE_LOG\n(DEMO.ML_DEMO)': (11, 4),
}

node_colors = {
    'CUSTOMER_LTV_PREDICTOR\n(MODEL V1)': '#d2a8ff',
    'CUSTOMER_FEATURES_FINAL\n(DEMO.ML_DEMO)': '#58a6ff',
    'CUSTOMER_FEATURES\n(DEMO.ML_DEMO)': '#58a6ff',
    'CUSTOMER\n(TPCH_SF1)': '#3fb950',
    'ORDERS\n(TPCH_SF1)'  : '#3fb950',
    'LINEITEM\n(TPCH_SF1)': '#3fb950',
    'NATION\n(TPCH_SF1)'  : '#3fb950',
    'INFERENCE_LOG\n(DEMO.ML_DEMO)': '#ffa657',
}

# 엣지 (화살표)
edges = [
    ('CUSTOMER_FEATURES_FINAL\n(DEMO.ML_DEMO)', 'CUSTOMER_LTV_PREDICTOR\n(MODEL V1)'),
    ('CUSTOMER_FEATURES\n(DEMO.ML_DEMO)', 'CUSTOMER_FEATURES_FINAL\n(DEMO.ML_DEMO)'),
    ('CUSTOMER\n(TPCH_SF1)',  'CUSTOMER_FEATURES\n(DEMO.ML_DEMO)'),
    ('ORDERS\n(TPCH_SF1)',   'CUSTOMER_FEATURES\n(DEMO.ML_DEMO)'),
    ('LINEITEM\n(TPCH_SF1)', 'CUSTOMER_FEATURES\n(DEMO.ML_DEMO)'),
    ('NATION\n(TPCH_SF1)',   'CUSTOMER_FEATURES\n(DEMO.ML_DEMO)'),
    ('INFERENCE_LOG\n(DEMO.ML_DEMO)', 'CUSTOMER_LTV_PREDICTOR\n(MODEL V1)'),
]

# 엣지 그리기
for src, dst in edges:
    x0, y0 = nodes[src]
    x1, y1 = nodes[dst]
    ax.annotate('', xy=(x1, y1), xytext=(x0, y0),
                arrowprops=dict(arrowstyle='->', color='#8b949e',
                                lw=1.5, connectionstyle='arc3,rad=0.05'))

# 노드 그리기
for name, (x_pos, y_pos) in nodes.items():
    color = node_colors[name]
    bbox = dict(boxstyle='round,pad=0.5', facecolor=color,
                edgecolor='white', alpha=0.15)
    ax.text(x_pos, y_pos, name, ha='center', va='center',
            color=color, fontsize=9, fontweight='bold', bbox=bbox)

# 레이어 레이블
layer_labels = [
    (0.3, 4.0, 'Layer 3\nModel', '#d2a8ff'),
    (0.3, 3.0, 'Layer 2\nFeature Table', '#58a6ff'),
    (0.3, 2.0, 'Layer 1\nTransformed', '#58a6ff'),
    (0.3, 0.8, 'Layer 0\nSource (TPC-H)', '#3fb950'),
]
for lx, ly, label, color in layer_labels:
    ax.text(lx, ly, label, ha='center', va='center',
            color=color, fontsize=8, style='italic', alpha=0.8)

ax.set_xlim(-0.5, 13)
ax.set_ylim(0, 5)
ax.set_title('ML Lineage Graph — CUSTOMER_LTV_PREDICTOR V1\n'
             '(모델 → 피처 테이블 → 원본 TPC-H 테이블)',
             color='#e6edf3', fontsize=13, fontweight='bold', pad=15)

# 범례
legend_items = [
    mpatches.Patch(facecolor='#d2a8ff', alpha=0.4, label='Model'),
    mpatches.Patch(facecolor='#58a6ff', alpha=0.4, label='Feature Table'),
    mpatches.Patch(facecolor='#3fb950', alpha=0.4, label='Source Table (TPC-H)'),
    mpatches.Patch(facecolor='#ffa657', alpha=0.4, label='Inference Log'),
]
ax.legend(handles=legend_items, loc='lower right',
          facecolor='#21262d', labelcolor='white', fontsize=9)

plt.tight_layout()
plt.savefig('/tmp/08_lineage_graph.png', dpi=150, bbox_inches='tight',
            facecolor='#0d1117')
plt.show()
print("\n✅ ML Lineage 그래프 시각화 완료")

In [ ]:
# ── Downstream Lineage: 모델 변경 영향 분석 ─────────────────────────────────
downstream_sql = """
SELECT *
FROM TABLE(
    DEMO.ML_DEMO.GET_LINEAGE(
        'MODEL',
        'DEMO.ML_DEMO.CUSTOMER_LTV_PREDICTOR/V1',
        'DOWNSTREAM',
        TRUE
    )
)
"""
print("📋 Downstream Lineage SQL (모델이 영향을 미치는 하위 객체):")
print(downstream_sql)

# 시뮬레이션: 하위 영향 객체
downstream_sim = pd.DataFrame([
    {'OBJECT_TYPE': 'TABLE',     'OBJECT_NAME': 'DEMO.ML_DEMO.INFERENCE_LOG',     'DISTANCE': 1},
    {'OBJECT_TYPE': 'VIEW',      'OBJECT_NAME': 'DEMO.ML_DEMO.MODEL_METRICS_DAILY','DISTANCE': 2},
    {'OBJECT_TYPE': 'DASHBOARD', 'OBJECT_NAME': 'Customer Value Dashboard',        'DISTANCE': 3},
    {'OBJECT_TYPE': 'ALERT',     'OBJECT_NAME': 'DEMO.ML_DEMO.MODEL_RMSE_ALERT','DISTANCE': 3},
])
print("\n📋 Downstream 영향 객체 (시뮬레이션):")
print(downstream_sim.to_string(index=False))
print("\n⚠️  모델 V1 → V2 업그레이드 시 위 모든 객체의 호환성을 확인해야 합니다!")

## 모듈 7 요약 (Summary)

---

### 학습한 내용

| 기능 | 도구 | 주요 포인트 |
|------|------|-------------|
| **추론 로그 관리** | Snowpark DataFrame | 예측 결과 + 실제 레이블 저장 |
| **Model Monitor** | `CREATE MODEL MONITOR` (SQL DDL) | 추론 로그 기반 자동 모니터링 |
| **성능 메트릭** | `MODEL_MONITOR_PERFORMANCE_METRIC()` | 일별 추이 시각화 |
| **피처 드리프트** | PSI (Population Stability Index) | PSI > 0.20 → 재학습 권고 |
| **자동 알림** | Snowflake Alerts | 임계값 기반 이메일/Slack 알림 |

---

### MLOps 전체 사이클

```
 ① 데이터 준비          ② 피처 엔지니어링     ③ 모델 학습
   TPC-H 테이블    →    Feature Store   →    XGBoost/LGB
                                                  ↓
 ⑥ 재학습 트리거        ⑤ 모니터링            ④ 모델 등록
   PSI > 0.20      ←   ML Observability ←   Model Registry
   Acc  < 0.85          (이 모듈!)              V1, V2...
```